
# Lab #7 — Keras MLP for Multiclass Classification

## Objective
Implement a Multi-Layer Perceptron (MLP) using Keras/TensorFlow for a multiclass classification problem and analyze the effect of different activation functions, optimizers, and model configurations on classification performance.

### Dataset
**Iris Dataset** — 150 observations, 4 numerical input features, and 3 target classes:
- Setosa
- Versicolor
- Virginica

The Iris dataset is suitable for this laboratory because it contains three classes and numerical features that can be directly scaled before MLP training.

### Experiments
Five models are compared using the **same train/test split and evaluation procedure**:

| Model | Hidden Activation | Optimizer |
|---|---|---|
| Model 1 | ReLU | Adam |
| Model 2 | Sigmoid | Adam |
| Model 3 | Tanh | Adam |
| Model 4 | ReLU | SGD |
| Model 5 | ReLU | RMSprop |

The comparison includes training/validation curves, test accuracy, precision, recall, F1-score, and confusion matrices.


## 1. Import Libraries

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


## 2. Load and Explore the Dataset

In [ ]:

iris = load_iris()

X = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)
y = pd.Series(iris.target, name="target")

class_names = iris.target_names

print("Dataset shape:", X.shape)
print("Number of input features:", X.shape[1])
print("Number of classes:", len(class_names))
print("Class names:", list(class_names))

display(X.head())
display(X.describe())

print("\nMissing values:")
display(X.isnull().sum())

print("\nClass distribution:")
class_counts = y.value_counts().sort_index()
class_distribution = pd.DataFrame({
    "Class": class_names,
    "Count": class_counts.values,
    "Percentage": (class_counts.values / len(y) * 100).round(2)
})
display(class_distribution)


In [ ]:

plt.figure(figsize=(7, 4))
sns.countplot(x=y.map(lambda i: class_names[i]))
plt.title("Iris Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.show()



### Dataset Description and Problem Definition

The Iris dataset contains measurements of iris flowers. The four input features are sepal length, sepal width, petal length, and petal width. The target variable is the iris species, with three classes: Setosa, Versicolor, and Virginica.

This is a **multiclass classification** problem because every observation belongs to one of three possible classes.

The dataset is balanced: each class contains 50 observations. There are no missing values in the supplied dataset, so no missing-value imputation is required.

Because the MLP is sensitive to the scale of numerical input features, standardization is applied before training.


## 3. Train/Test Split and Feature Scaling

In [ ]:

# Same split is reused for every experiment for a fair comparison.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

# Fit scaler ONLY on training data to avoid data leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train_scaled.shape[0])
print("Testing samples:", X_test_scaled.shape[0])
print("Scaled training feature means:", np.round(X_train_scaled.mean(axis=0), 4))
print("Scaled training feature standard deviations:", np.round(X_train_scaled.std(axis=0), 4))



### Preprocessing Justification

- **Stratified train/test split:** preserves the class proportions in both subsets.
- **StandardScaler:** transforms each numerical feature to approximately zero mean and unit variance, helping the neural network optimize more consistently.
- **No missing-value treatment:** the dataset contains no missing values.
- **No categorical feature encoding:** all four input features are numerical.
- **Target encoding:** the target is already represented as integer class labels (0, 1, 2). Sparse categorical cross-entropy allows these labels to be used directly.
- **Validation strategy:** 20% of the training data is used as validation data during each model's training. The test set remains unseen until final evaluation.


## 4. MLP Architecture


The common architecture used for all experiments is:

**Input (4 features) → Dense(16) → Dense(8) → Output(3, Softmax)**

- Input features: **4**
- Hidden layer 1: **16 neurons**
- Hidden layer 2: **8 neurons**
- Hidden-layer activation: varied between ReLU, Sigmoid, and Tanh
- Output layer: **3 neurons**
- Output activation: **Softmax**
- Loss: **Sparse Categorical Cross-Entropy**
- Batch size: **16**
- Epochs: **100**
- Validation split: **20% of the training set**

Only the activation function or optimizer is changed between the designated experiments so that the comparison remains fair.


In [ ]:

EPOCHS = 100
BATCH_SIZE = 16
VALIDATION_SPLIT = 0.20

def build_mlp(activation="relu", optimizer=None):
    model = Sequential([
        Input(shape=(X_train_scaled.shape[1],)),
        Dense(16, activation=activation),
        Dense(8, activation=activation),
        Dense(len(class_names), activation="softmax")
    ])

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


## 5. Experiment Setup

In [ ]:

experiments = {
    "Model 1 - ReLU + Adam": {
        "activation": "relu",
        "optimizer": Adam(learning_rate=0.001)
    },
    "Model 2 - Sigmoid + Adam": {
        "activation": "sigmoid",
        "optimizer": Adam(learning_rate=0.001)
    },
    "Model 3 - Tanh + Adam": {
        "activation": "tanh",
        "optimizer": Adam(learning_rate=0.001)
    },
    "Model 4 - ReLU + SGD": {
        "activation": "relu",
        "optimizer": SGD(learning_rate=0.01)
    },
    "Model 5 - ReLU + RMSprop": {
        "activation": "relu",
        "optimizer": RMSprop(learning_rate=0.001)
    }
}

histories = {}
models = {}
results = {}
confusion_matrices = {}


## 6. Train All Five Models

In [ ]:

for name, config in experiments.items():
    print("=" * 70)
    print(name)

    # Reset seed before each model to improve reproducibility.
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model = build_mlp(
        activation=config["activation"],
        optimizer=config["optimizer"]
    )

    history = model.fit(
        X_train_scaled,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        verbose=0
    )

    models[name] = model
    histories[name] = history

    print("Final training accuracy:",
          round(history.history["accuracy"][-1], 4))
    print("Final validation accuracy:",
          round(history.history["val_accuracy"][-1], 4))
    print("Minimum validation loss:",
          round(min(history.history["val_loss"]), 4))


## 7. Training and Validation Curves

In [ ]:

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(history.history["accuracy"], label="Training Accuracy")
    axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy")
    axes[0].set_title(f"{title} - Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(history.history["loss"], label="Training Loss")
    axes[1].plot(history.history["val_loss"], label="Validation Loss")
    axes[1].set_title(f"{title} - Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

for name, history in histories.items():
    plot_history(history, name)



### Learning Curve Interpretation

The curves should be examined for:

- **Overfitting:** training performance continues improving while validation performance stops improving or validation loss rises.
- **Underfitting:** both training and validation performance remain relatively poor.
- **Poor convergence:** the curves remain unstable or improve very slowly.
- **Stable convergence:** training and validation performance improve and then stabilize with a relatively small gap.

The exact behavior can vary because neural-network optimization is iterative. The final interpretation below is generated from the actual results rather than assuming that one activation function or optimizer must always win.


## 8. Evaluate Every Model on the Unseen Test Set

In [ ]:

for name, model in models.items():
    y_prob = model.predict(X_test_scaled, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    results[name] = {
        "Activation": experiments[name]["activation"].capitalize(),
        "Optimizer": type(experiments[name]["optimizer"]).__name__,
        "Training Accuracy": histories[name].history["accuracy"][-1],
        "Validation Accuracy": histories[name].history["val_accuracy"][-1],
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (Macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-Score (Macro)": f1_score(y_test, y_pred, average="macro", zero_division=0)
    }

    confusion_matrices[name] = confusion_matrix(y_test, y_pred)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("Test Accuracy", ascending=False)

display(results_df.round(4))



### Evaluation Metric Choice

Because this is a multiclass problem, **macro-average precision, recall, and F1-score** are reported. Macro averaging gives every class equal importance and is appropriate for examining performance across the three classes.

Test metrics are calculated only on the unseen test set.


## 9. Confusion Matrices

In [ ]:

for name, cm in confusion_matrices.items():
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.show()


## 10. Detailed Classification Reports

In [ ]:

for name, model in models.items():
    y_pred = np.argmax(model.predict(X_test_scaled, verbose=0), axis=1)

    print("=" * 80)
    print(name)
    print(classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    ))


## 11. Required Comparison Table

In [ ]:

comparison_table = results_df[
    [
        "Activation", "Optimizer",
        "Test Accuracy", "Precision (Macro)",
        "Recall (Macro)", "F1-Score (Macro)"
    ]
].copy()

display(comparison_table.round(4))


## 12. Convergence Comparison

In [ ]:

convergence_rows = []

for name, history in histories.items():
    val_loss = np.array(history.history["val_loss"])
    val_acc = np.array(history.history["val_accuracy"])

    best_epoch = int(np.argmin(val_loss)) + 1

    convergence_rows.append({
        "Model": name,
        "Best Validation Loss": val_loss.min(),
        "Epoch of Best Val Loss": best_epoch,
        "Best Validation Accuracy": val_acc.max(),
        "Final Validation Accuracy": val_acc[-1]
    })

convergence_df = pd.DataFrame(convergence_rows).sort_values(
    "Best Validation Accuracy", ascending=False
)

display(convergence_df.round(4))


## 13. Automatic Identification of Best Model

In [ ]:

best_model_name = results_df.index[0]
best_model = models[best_model_name]

best_row = results_df.loc[best_model_name]

best_y_pred = np.argmax(
    best_model.predict(X_test_scaled, verbose=0),
    axis=1
)

best_cm = confusion_matrices[best_model_name]

print("BEST MODEL:", best_model_name)
print()
print("Test Accuracy:", round(best_row["Test Accuracy"], 4))
print("Macro Precision:", round(best_row["Precision (Macro)"], 4))
print("Macro Recall:", round(best_row["Recall (Macro)"], 4))
print("Macro F1:", round(best_row["F1-Score (Macro)"], 4))

print("\nConfusion Matrix:")
print(best_cm)


## 14. Class-Level Error Analysis

In [ ]:

# Correct predictions per actual class
correct_by_class = np.diag(best_cm)
total_by_class = best_cm.sum(axis=1)
class_accuracy = correct_by_class / total_by_class

class_accuracy_df = pd.DataFrame({
    "Class": class_names,
    "Correct": correct_by_class,
    "Total": total_by_class,
    "Class Accuracy": class_accuracy
})

display(class_accuracy_df.round(4))

most_accurately_predicted = class_names[np.argmax(class_accuracy)]

# Find the most frequent off-diagonal confusion.
cm_errors = best_cm.copy()
np.fill_diagonal(cm_errors, 0)

max_error = cm_errors.max()
error_pairs = np.argwhere(cm_errors == max_error)

print("Most accurately predicted class:", most_accurately_predicted)

if max_error > 0:
    pairs = [
        f"{class_names[actual]} → {class_names[predicted]}"
        for actual, predicted in error_pairs
    ]
    print("Most frequent misclassification:", ", ".join(pairs))
    print("Number of such misclassifications:", int(max_error))
else:
    print("No misclassifications occurred for the selected model.")



## 15. Analysis and Interpretation

### 1. Which activation function performed best?
The answer should be based on the comparison table rather than assuming that one activation is universally superior. ReLU is commonly effective in hidden layers because it is simple and generally provides strong gradients for positive inputs. Sigmoid can converge differently because it saturates at large positive or negative values. Tanh is zero-centered and can behave differently from both ReLU and sigmoid.

### 2. Which optimizer performed best?
The optimizer should be selected using test performance together with validation behavior and convergence. Adam and RMSprop use adaptive learning-rate mechanisms, while SGD uses a more direct gradient-descent update. Their behavior can therefore differ in convergence speed and stability.

### 3. Which activation-function/optimizer combination produced the best overall performance?
The notebook selects the model with the highest test accuracy in the results table. Precision, recall, F1-score, validation behavior, and the confusion matrix should also be considered before making the final decision.

### 4. Did activation function significantly affect convergence?
Compare the learning curves of ReLU, sigmoid, and tanh while the optimizer is held at Adam. Differences in the rate at which training/validation loss and accuracy stabilize indicate the effect of the activation function.

### 5. Did optimizer affect convergence speed?
Compare ReLU + Adam, ReLU + SGD, and ReLU + RMSprop. The epoch at which validation loss reaches its minimum provides one useful indicator of convergence.

### 6. Which model achieved the best validation performance?
This can be obtained from the convergence table using the highest validation accuracy.

### 7. Which model achieved the best test performance?
The model at the top of the comparison table has the highest test accuracy.

### 8. Is there a significant difference between training and testing performance?
Compare the training accuracy with test accuracy. A large gap can indicate overfitting or a difference between training and unseen-data performance.

### 9. Do the learning curves indicate overfitting or underfitting?
Use the plotted training/validation curves. A widening performance gap or rising validation loss while training loss continues falling is evidence of overfitting.

### 10. Which classes were most frequently misclassified?
The class-level analysis and confusion matrix identify the actual/predicted class pair with the largest off-diagonal count.

### 11. Possible reasons for misclassification
For Iris, some classes are more difficult to separate because their feature measurements overlap more strongly than those of clearly separated classes. In particular, classes with similar petal/sepal measurements can produce boundary ambiguity.

### 12. What changes could improve classification performance?
Possible changes include tuning the number of hidden neurons, learning rate, batch size, epochs, optimizer settings, regularization, dropout, or the train/validation strategy. Since this dataset is small, overly complex architectures should be avoided.

### 13. Which model should be selected as the final model?
Select the model using the combined evidence from test accuracy, macro precision/recall/F1, confusion matrix, validation curves, convergence, complexity, and generalization—not accuracy alone.


## 16. Final Model Selection

In [ ]:

# A simple evidence-based selection:
# First rank by test F1, then test accuracy.
# This avoids relying exclusively on accuracy.
selection_df = results_df.copy()
selection_df["Selection Score"] = (
    0.5 * selection_df["F1-Score (Macro)"] +
    0.5 * selection_df["Test Accuracy"]
)
selection_df = selection_df.sort_values(
    ["Selection Score", "Test Accuracy"],
    ascending=False
)

selected_name = selection_df.index[0]
selected_row = selection_df.loc[selected_name]

print("FINAL SELECTED MODEL:", selected_name)
print()
print("Reasoning based on the experimental evidence:")
print(f"- Test accuracy: {selected_row['Test Accuracy']:.4f}")
print(f"- Macro precision: {selected_row['Precision (Macro)']:.4f}")
print(f"- Macro recall: {selected_row['Recall (Macro)']:.4f}")
print(f"- Macro F1-score: {selected_row['F1-Score (Macro)']:.4f}")
print("- The final choice should also be checked against its learning curves and confusion matrix.")



### Final Model Justification

The final model is selected using multiple forms of experimental evidence rather than test accuracy alone. The decision considers:

1. Test accuracy
2. Macro precision
3. Macro recall
4. Macro F1-score
5. Confusion matrix
6. Training/validation curves
7. Convergence behavior
8. Model complexity
9. Generalization performance

The selected architecture remains relatively small because the Iris dataset contains only 150 observations. A substantially larger network would add complexity without necessarily providing a meaningful benefit.



## 17. Conclusion

This laboratory implemented a Keras/TensorFlow MLP for multiclass classification using the Iris dataset. The model used four numerical input features and predicted three flower species.

Five configurations were evaluated:
- ReLU + Adam
- Sigmoid + Adam
- Tanh + Adam
- ReLU + SGD
- ReLU + RMSprop

The experiments demonstrate that both hidden-layer activation functions and optimizers can influence learning dynamics, convergence, and generalization. The training/validation curves provide evidence about convergence and possible overfitting, while the test metrics and confusion matrices provide an evaluation on unseen data.

The final model was selected using multiple performance measures and model behavior rather than relying solely on accuracy. This satisfies the requirement that the experiment include comparison, validation, interpretation, and evidence-based model selection.



## 18. Requirements Checklist

- [x] Dataset description and source
- [x] Problem definition
- [x] Dataset exploration
- [x] Class distribution analysis
- [x] Missing-value analysis
- [x] Feature scaling
- [x] Target encoding
- [x] Train/test split
- [x] Validation strategy
- [x] MLP with at least two hidden layers
- [x] Softmax multiclass output
- [x] ReLU experiment
- [x] Sigmoid experiment
- [x] Tanh experiment
- [x] SGD experiment
- [x] Adam experiment
- [x] RMSprop experiment
- [x] Training accuracy/loss
- [x] Validation accuracy/loss
- [x] Test accuracy
- [x] Precision
- [x] Recall
- [x] F1-score
- [x] Confusion matrices
- [x] Learning curves
- [x] Comparison table
- [x] Convergence analysis
- [x] Misclassification analysis
- [x] Final model selection
- [x] Justification and conclusion
